**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->
# Computer Vision — Lezione 19
## Visione multimodale e generativa
23 novembre 2026. CLIP reale su tre immagini pubbliche e calcoli analitici sui meccanismi della diffusione. Cxx conta tutte le celle. Nessun generatore text-to-image viene addestrato o eseguito.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

<!-- C02 -->
## Avvio Colab e materiali
Aprire questo notebook in Colab. Caricare anche `Pacchetto_studenti_Lezione_19.zip` nello **stesso runtime**: C03 offre il caricamento se i file mancano. CPU è sufficiente. La prima inferenza LIVE in C11 scarica circa 338 MiB di pesi CLIP pubblici e piccole dipendenze. Farla prima dell’esecuzione, oppure impostare `MODE="riferimento"` in C03 per leggere le feature della prova CPU già svolta. Questa modalità non è una nuova inferenza. I calcoli successivi vengono comunque rieseguiti. La prova locale non certifica una sessione del servizio Google Colab. Salvare notebook e ZIP output con C31 prima di chiudere il runtime. Non occorre montare Drive.

In [ ]:
# C03
import os,sys,json,time,hashlib,zipfile,subprocess,importlib.util
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
MODE=os.environ.get("CV19_MODE","live") # "live" oppure "riferimento"
BASE=Path(os.environ.get("CV19_BASE","." )).resolve()
def lesson19_root(folder):
    try:
        m=json.loads((folder/"input/manifest.json").read_text())
        names={it["file"] for it in m["images"]}
        return (m.get("lesson")==19 and names=={"astronaut.png","coffee.png","chelsea.png"}
                and all((folder/"input"/name).is_file() for name in names)
                and (folder/"riferimento/clip_features.npz").is_file())
    except (OSError,ValueError,KeyError,TypeError):
        return False
if not lesson19_root(BASE):
    candidates=sorted({p.parent.parent for p in BASE.glob("**/input/manifest.json") if lesson19_root(p.parent.parent)})
    if not candidates and lesson19_root(BASE.parent):candidates=[BASE.parent]
    if len(candidates)>1:raise ValueError("Più pacchetti Lezione 19: impostare CV19_BASE sulla cartella desiderata")
    if candidates: BASE=candidates[0]
    elif IN_COLAB:
        from google.colab import files
        uploaded=files.upload()
        zips=[(n,b) for n,b in uploaded.items() if n.lower().endswith(".zip")]
        if len(zips)!=1: raise ValueError("Caricare un solo Pacchetto_studenti_Lezione_19.zip")
        import io,tempfile
        target=Path(tempfile.mkdtemp(prefix="materiali_lezione19_",dir=".")).resolve()
        with zipfile.ZipFile(io.BytesIO(zips[0][1])) as z:
            for info in z.infolist():
                dst=(target/info.filename).resolve()
                if not dst.is_relative_to(target): raise ValueError("Percorso ZIP non valido")
            z.extractall(target)
        candidates=sorted({p.parent.parent for p in target.glob("**/input/manifest.json") if lesson19_root(p.parent.parent)})
        if len(candidates)!=1:raise ValueError("ZIP errato o incompleto: caricare Pacchetto_studenti_Lezione_19.zip con manifest lesson=19, tre immagini e feature CLIP")
        BASE=candidates[0]
    else: raise FileNotFoundError("Materiali Lezione 19 mancanti o incompleti: estrarre Pacchetto_studenti_Lezione_19.zip e avviare dalla sua cartella")
assert MODE in ("live","riferimento")
manifest=json.loads((BASE/"input/manifest.json").read_text())
for it in manifest["images"]:
    assert hashlib.sha256((BASE/"input"/it["file"]).read_bytes()).hexdigest()==it["sha256"]
OUT=Path("output_lezione19").resolve();OUT.mkdir(exist_ok=True)
RESULTS={"mode":MODE,"numpy":np.__version__,"python":sys.version.split()[0],"protocollo":"3 immagini illustrative, non benchmark; calcoli analitici CPU"}
np.set_printoptions(precision=6,suppress=True);plt.rcParams.update({"font.size":13,"figure.dpi":120})
def unit(x):
    x=np.asarray(x,float);d=np.linalg.norm(x,axis=-1,keepdims=True)
    if np.any(d==0):raise ValueError("Norma zero")
    return x/d
def softmax(x,axis=-1):
    x=np.asarray(x,float);e=np.exp(x-x.max(axis=axis,keepdims=True));return e/e.sum(axis=axis,keepdims=True)
def figsave(name):
    plt.tight_layout();plt.savefig(OUT/(name+".png"),dpi=180,bbox_inches="tight");plt.show()
print("Materiali",BASE,"modalità",MODE,"CPU per calcoli analitici")


<!-- C04 -->
## N01 — Normalizzazione e coseno
C05 richiede C03. Con u=(3, 4), t1=(0, 5), t2=(-3, -4), confrontare prodotto scalare e coseno. Moltiplicare u per 10 cambia il ranking dei coseni?

In [ ]:
# C05
u=np.array([3.,4.]);texts=np.array([[0.,5.],[-3.,-4.]])
cos=unit(u)@unit(texts).T
assert np.allclose(cos,[.8,-1]) and np.allclose(cos,unit(10*u)@unit(texts).T)
print("prodotti scalari",u@texts.T,"coseni",cos)
RESULTS["coseno"]={"dot":(u@texts.T).tolist(),"cosine":cos.tolist()}


<!-- C06 -->
## N02 — Loss contrastiva simmetrica

**Notazione.** $s_{ij}$ è il coseno immagine-testo; $z_{ij}=\kappa s_{ij}$ è il logit scalato passato alla cross-entropy, con scala positiva $\kappa$ e temperatura $1/\kappa$. Questi indici riguardano abbinamenti, mentre nelle lezioni precedenti $s$ indicava score di classe.

C07 richiede C03. Usare la matrice di similarità [[1, 0], [0, 1]] e scala 2. Le coppie corrette sono sulla diagonale. Calcolare le CE lungo righe e colonne. Cosa cambia con due caption equivalenti trattate come negative?

In [ ]:
# C07
S=np.eye(2);logits=2*S;pi=softmax(logits,1);pt=softmax(logits,0)
li=-np.log(np.diag(pi)).mean();lt=-np.log(np.diag(pt)).mean();loss=(li+lt)/2
print("probabilità image→text",pi,"L image/text/simmetrica",li,lt,loss,sep="\n")
assert np.isclose(loss,np.log1p(np.exp(-2)))
RESULTS["contrastiva"]={"S":S.tolist(),"logits":logits.tolist(),"loss_i":float(li),"loss_t":float(lt),"loss":float(loss)}


<!-- C08 -->
## N03 — Temperatura e vocabolario candidato
C09 richiede C03. Con similarità (0.3, 0.2) confrontare scale 1 e10. Aggiungere una terza classe con similarità0.29. La softmax è una probabilità di correttezza assoluta?

In [ ]:
# C09
a=softmax(np.array([.3,.2]));b=softmax(10*np.array([.3,.2]));c=softmax(10*np.array([.3,.2,.29]))
print("scala1",a,"scala10",b,"classe aggiunta",c)
RESULTS["temperatura"]={"scale1":a.tolist(),"scale10":b.tolist(),"added_candidate":c.tolist()}


<!-- C10 -->
## N04 — CLIP reale e preprocessing
C11 richiede C03. Inferenza OpenAI CLIP ViT-B/32, encoder congelati. Tre fotografie: astronauta, caffè, gatto. Quattro classi candidate: astronaut, coffee, cat, dog. Si registrano feature normalizzate, scala appresa, tempi e hash dei materiali. Perché queste tre immagini non costituiscono una stima affidabile di accuratezza generale?

In [ ]:
# C11
classes=["astronaut","coffee","cat","dog"]
templates=["{}","a photo of {}","a blurry photo of {}"]
descriptions=["an astronaut","a cup of coffee","a cat","a dog"]
image_files=["astronaut.png","coffee.png","chelsea.png"]
prompts=[c if j==0 else t.format(descriptions[i]) for i,c in enumerate(classes) for j,t in enumerate(templates)]
ref=BASE/"riferimento/clip_features.npz"
if MODE=="live":
    import torch
    if importlib.util.find_spec("clip") is None:
        subprocess.check_call([sys.executable,"-m","pip","-q","install","ftfy","regex","tqdm","git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"])
    import clip
    torch.set_num_threads(1)
    device=os.environ.get("CV19_DEVICE","cpu")
    if device not in ("cpu","cuda"):raise ValueError("Dispositivo ammesso: cpu o cuda")
    if device=="cuda" and not torch.cuda.is_available():raise RuntimeError("GPU CUDA non disponibile")
    checkpoint=BASE/"models/ViT-B-32.pt"
    t0=time.perf_counter()
    model,preprocess=clip.load(str(checkpoint) if checkpoint.exists() else "ViT-B/32",device=device,download_root=str(BASE/"models"))
    model.eval();load_s=time.perf_counter()-t0
    actual_checkpoint=checkpoint if checkpoint.exists() else BASE/"models/ViT-B-32.pt"
    measured_hash=hashlib.sha256(actual_checkpoint.read_bytes()).hexdigest()
    assert measured_hash=="40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af"
    images=torch.stack([preprocess(Image.open(BASE/"input"/f).convert("RGB")) for f in image_files]).to(device)
    tokens=clip.tokenize(prompts).to(device)
    if device=="cuda":torch.cuda.synchronize()
    t0=time.perf_counter()
    with torch.inference_mode():
        image_features=model.encode_image(images).float();text_features=model.encode_text(tokens).float()
        image_features/=image_features.norm(dim=-1,keepdim=True);text_features/=text_features.norm(dim=-1,keepdim=True)
        scale=float(model.logit_scale.exp().item())
    if device=="cuda":torch.cuda.synchronize()
    inference_s=time.perf_counter()-t0
    image_features=image_features.cpu().numpy();text_features=text_features.cpu().numpy()
    clip_meta={"execution":"inferenza reale locale o nel runtime attuale","device":device,"dtype":"float32","threads":1,"torch":torch.__version__,"load_s":load_s,"inference_s":inference_s,"checkpoint_sha256":measured_hash}
else:
    cache=np.load(ref,allow_pickle=False);image_features=cache["image_features"];text_features=cache["text_features"];scale=float(cache["scale"])
    clip_meta=json.loads((BASE/"riferimento/clip_meta.json").read_text());clip_meta={**clip_meta,"execution":"lettura feature della prova CPU già eseguita; nessuna nuova inferenza"}
assert image_features.shape==(3,512) and text_features.shape==(12,512)
np.savez_compressed(OUT/"clip_features.npz",image_features=image_features,text_features=text_features,scale=scale)
(OUT/"clip_meta.json").write_text(json.dumps(clip_meta,indent=2))
RESULTS["clip"]={"meta":clip_meta,"scale":scale,"classes":classes,"prompts":prompts,"images":image_files}
print("Feature",image_features.shape,text_features.shape,"scala",scale,clip_meta)


<!-- C12 -->
## N05 — Prompt, ensemble e categorie mancanti
C13 richiede C03/C11. Confrontare nome isolato, frase fotografica e media dei tre embedding testuali normalizzati, rinormalizzata. Togliere cat ai candidati e osservare la decisione sull’immagine del gatto. Quale test servirebbe per rifiutare un’immagine sconosciuta?

In [ ]:
# C13
tf=text_features.reshape(4,3,512);tf_ens=unit(tf.mean(1))
scores={"nome":image_features@tf[:,0].T,"foto":image_features@tf[:,1].T,"ensemble":image_features@tf_ens.T}
for name,s in scores.items():print(name,"coseni",s,"predizioni",[classes[k] for k in s.argmax(1)],sep="\n")
no_cat=[0,1,3];forced=no_cat[int(scores["foto"][2,no_cat].argmax())]
print("Gatto senza cat:",classes[forced],"coseni",scores["foto"][2,no_cat])
RESULTS["prompts"]={name:{"cosine":s.tolist(),"predictions":[classes[k] for k in s.argmax(1)],"softmax":softmax(scale*s).tolist()} for name,s in scores.items()}
RESULTS["missing_class"]={"candidates":[classes[k] for k in no_cat],"prediction":classes[forced],"cosine":scores["foto"][2,no_cat].tolist()}
fig,ax=plt.subplots(figsize=(9,3.8));im=ax.imshow(scores["foto"],vmin=0,vmax=.4,cmap="viridis");ax.set(xticks=range(4),xticklabels=classes,yticks=range(3),yticklabels=["astronauta","caffè","gatto"],xlabel="Testo candidato",ylabel="Immagine")
for i in range(3):
    for j in range(4):ax.text(j,i,f"{scores['foto'][i,j]:.3f}",ha="center",va="center",color="white")
fig.colorbar(im,ax=ax,label="Coseno");figsave("clip_coseni")


<!-- C14 -->
## N06 — Retrieval in due direzioni
C15 richiede C03/C11/C13. Ordinare i testi per immagine e le immagini per testo. Un cane può essere recuperato se nessuna delle tre fotografie mostra un cane? Quali limiti ha Recall@1 su questa galleria?

In [ ]:
# C15
s=scores["foto"];image_to_text=np.argsort(-s,axis=1);text_to_image=np.argsort(-s,axis=0)
print("ranking testi per immagine",image_to_text,"ranking immagini per testo",text_to_image,sep="\n")
RESULTS["retrieval"]={"image_to_text":image_to_text.tolist(),"text_to_image":text_to_image.tolist(),"recall1_known3":float(np.mean(np.argmax(s[:,:3],axis=0)==np.arange(3)))}


<!-- C16 -->
## N07 — Schedule della diffusione

**Prima del calcolo.** La schedule è la sequenza delle varianze $\beta_t$; $\bar\alpha_t$ è un prodotto cumulativo. SNR nominale indica il rapporto $\bar\alpha_t/(1-\bar\alpha_t)$ per segnale a varianza unitaria. Nella catena Markov il passo usa lo stato precedente con nuovo rumore indipendente. $N(0,I)$ indica componenti gaussiane standard, media zero e covarianza identità.

C17 richiede C03. Con beta=(0.1, 0.2, 0.3), calcolare alpha, prodotto cumulativo e SNR. Distinguere beta_t dalla varianza marginale 1-alpha_bar_t.

In [ ]:
# C17
beta=np.array([.1,.2,.3]);alpha=1-beta;abar=np.cumprod(alpha);snr=abar/(1-abar)
print("beta",beta,"alpha",alpha,"alpha_bar",abar,"SNR",snr,sep="\n")
assert np.allclose(abar,[.9,.72,.504])
RESULTS["schedule"]={"beta":beta.tolist(),"alpha":alpha.tolist(),"alpha_bar":abar.tolist(),"snr":snr.tolist()}


<!-- C18 -->
## N08 — Marginale diretta e rumore nell’immagine
C19 richiede C03/C17. Calcolare x_t per x0=1, epsilon=-0.5, alpha_bar=0.64. Visualizzare tre livelli di corruzione della fotografia, mantenendo lo stesso epsilon per confrontare i marginali. È una singola traiettoria Markov?

In [ ]:
# C19
x0=1.;eps=-.5;at=.64;xt=np.sqrt(at)*x0+np.sqrt(1-at)*eps
assert np.isclose(xt,.5)
rng=np.random.default_rng(19);photo=np.asarray(Image.open(BASE/"input/coffee.png").convert("RGB").resize((240,160)),float)/127.5-1;noise=rng.standard_normal(photo.shape)
fig,axes=plt.subplots(1,3,figsize=(12,3.6))
for ax,a in zip(axes,[.9,.5,.05]):
    noisy=np.sqrt(a)*photo+np.sqrt(1-a)*noise;ax.imshow(np.clip((noisy+1)/2,0,1));ax.set_title(f"alpha_bar = {a}");ax.axis("off")
figsave("forward_marginali")
RESULTS["forward"]={"x0":x0,"epsilon":eps,"alpha_bar":at,"xt":float(xt),"visualization":"epsilon condiviso, marginali correlati, non traiettoria Markov"}


<!-- C20 -->
## N09 — Ricostruzione oracle e amplificazione dell’errore

**Informazione disponibile.** Un oracle conosce il rumore esatto usato nella corruzione e può verificare l’identità analitica. Nel modello appreso quel rumore è un target di training: la rete riceve il dato rumoroso e il tempo, non il target.

C21 richiede C03/C19. Ricostruire x0 usando epsilon esatto. Poi usare epsilon_hat=-0.4 invece di-0.5. Come dipende l’errore dal rumore? Questo oracle è disponibile in generazione?

In [ ]:
# C21
oracle=(xt-np.sqrt(1-at)*eps)/np.sqrt(at);eps_hat=-.4;xhat=(xt-np.sqrt(1-at)*eps_hat)/np.sqrt(at)
a_grid=np.array([.9,.64,.1,.01]);ampl=np.sqrt((1-a_grid)/a_grid)
print("oracle",oracle,"stima",xhat,"errore",xhat-x0,"amplificazione",ampl)
RESULTS["reconstruction"]={"oracle":float(oracle),"estimate":float(xhat),"error":float(xhat-x0),"alpha_bar":a_grid.tolist(),"amplification":ampl.tolist()}


<!-- C22 -->
## N10 — Media inversa DDPM

**DDPM** significa *Denoising Diffusion Probabilistic Model*. $\mu_\theta$ è la media inversa stimata; $\sigma_t$ è una deviazione standard, $\sigma_t^2$ una varianza. La varianza del posterior con dato pulito noto va distinta dal reverse usato per generare senza conoscere quel dato.

C23 richiede C03. Per alpha=0.9, beta=0.1, alpha_bar=0.64, x_t=0.5 ed epsilon_hat=-0.4 calcolare mu_theta. Qual è la differenza tra questa media e x0_hat?

In [ ]:
# C23
alpha_t=.9;beta_t=.1;abar_t=.64;xt_t=.5;eh=-.4
mu=(xt_t-beta_t/np.sqrt(1-abar_t)*eh)/np.sqrt(alpha_t)
abar_prev=abar_t/alpha_t;posterior_var=beta_t*(1-abar_prev)/(1-abar_t)
print("mu_theta",mu,"beta_tilde",posterior_var)
RESULTS["reverse_step"]={"mean":float(mu),"posterior_variance":float(posterior_var),"alpha_bar_previous":float(abar_prev)}


<!-- C24 -->
## N11 — Campionamento con reverse gaussiano analitico
C25 richiede C03. Si conosce p_data=N(2, 0.5²), non si addestra una rete. Si costruisce il reverse condizionale esatto di una catena lineare gaussiana. Verificare media e deviazione finale di 4000 campioni. Perché il rumore iniziale va estratto dalla marginale finale esatta in questa prova?

In [ ]:
# C25
rng=np.random.default_rng(1919);T=50;b=np.linspace(.01,.15,T);a=1-b;ab=np.r_[1.,np.cumprod(a)]
m=2*np.sqrt(ab);v=.25*ab+1-ab;count=4000
samples=rng.normal(m[-1],np.sqrt(v[-1]),size=count);initial=samples.copy()
for t in range(T,0,-1):
    cov=np.sqrt(a[t-1])*v[t-1];gain=cov/v[t]
    mean=m[t-1]+gain*(samples-m[t]);var=max(v[t-1]-cov*cov/v[t],0)
    samples=mean+np.sqrt(var)*rng.standard_normal(count)
print("media",samples.mean(),"std",samples.std(ddof=1),"target",2.,.5)
assert abs(samples.mean()-2)<.05 and abs(samples.std(ddof=1)-.5)<.03
fig,ax=plt.subplots(figsize=(10,3.7));ax.hist(initial,bins=55,density=True,alpha=.45,label="Marginale iniziale t=T");ax.hist(samples,bins=55,density=True,alpha=.6,label="Campioni dopo reverse");xx=np.linspace(-4,5,400);density=np.exp(-.5*((xx-2)/.5)**2)/(.5*np.sqrt(2*np.pi));ax.plot(xx,density,label="Densità target nota");ax.set(xlabel="Valore scalare x",ylabel="Densità");ax.legend();figsave("reverse_gaussiano")
RESULTS["gaussian_reverse"]={"n":count,"steps":T,"mean":float(samples.mean()),"std":float(samples.std(ddof=1)),"target_mean":2.,"target_std":.5,"prior_mean":float(m[-1]),"prior_var":float(v[-1]),"trained":False}


<!-- C26 -->
## N12 — Classifier-free guidance

**CFG** significa *classifier-free guidance*, senza classificatore esterno. Il coefficiente $s$ di questa cella è distinto dal coseno $s_{ij}$: combina due stime dello stesso denoiser, con e senza condizione.

C27 richiede C03. Dati epsilon_u=(1, 0), epsilon_c=(0.6, 0.2), calcolare epsilon_u+s(epsilon_c−epsilon_u) con s=0, 1, 3. Quale valore restituisce la previsione condizionata senza extrapolazione?

In [ ]:
# C27
eu=np.array([1.,0.]);ec=np.array([.6,.2]);scales=[0,1,3];guided=np.stack([eu+s*(ec-eu) for s in scales])
print("scale",scales,"epsilon guidato",guided)
assert np.allclose(guided[-1],[-.2,.6])
RESULTS["cfg"]={"scale":scales,"epsilon_uncond":eu.tolist(),"epsilon_cond":ec.tolist(),"guided":guided.tolist()}


<!-- C28 -->
## N13 — Protocollo per dati sintetici
C29 richiede C03. Su un test reale fisso, baseline80/100 e modello con dati sintetici84/100. Il solo aumento di accuracy prova che i dati sintetici migliorano sempre? Definire quali informazioni mancano.

In [ ]:
# C29
baseline_correct=80;synthetic_correct=84;n=100
print("accuracy baseline",baseline_correct/n,"con sintetici",synthetic_correct/n,"delta punti percentuali",100*(synthetic_correct-baseline_correct)/n)
RESULTS["synthetic_protocol"]={"assigned_example":True,"baseline_accuracy":.8,"synthetic_accuracy":.84,"delta_percentage_points":4.,"paired_outcomes_available":False}


<!-- C30 -->
## Esportazione e consegna
C31 richiede C03 e tutti i blocchi eseguiti. Salvare i risultati JSON, le figure e le feature in ZIP. Consegna: F01 spiegare la loss e i candidati, F02 commentare prompt e categoria mancante, F03 distinguere oracle e generatore, F04 interpretare CFG, F05 progettare una valutazione dei sintetici. Non consegnare pesi CLIP.

In [ ]:
# C31
(OUT/"risultati_lezione19.json").write_text(json.dumps(RESULTS,indent=2,ensure_ascii=False))
archive=OUT.parent/"Output_Lezione_19.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,p.name)
print("Salvato",archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))


<!-- C32 -->
## Ripresa senza download
Se il runtime si interrompe: rieseguire C03 con MODE="riferimento" e poi C11, C13, C15 per CLIP. Per i blocchi analitici rieseguire C03 e le dipendenze indicate. C33 mostra il JSON della prova distribuita: lettura di risultati precedenti, senza nuova inferenza o misura dei tempi. Per rigenerare gli output eseguire i blocchi completi e C31.

In [ ]:
# C33
saved=json.loads((BASE/"riferimento/risultati_lezione19.json").read_text())
print("RECUPERO: risultati della prova locale CPU precedente")
print("Predizioni",saved["prompts"]["foto"]["predictions"])
print("Gaussian reverse",saved["gaussian_reverse"])
